# Instructor Effectiveness Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('instructor_effectiveness_dataset_2000_rows - instructor_effectiveness_dataset_2000_rows.csv.csv')
df.head()

Let's check the data info and see if there are any missing values.

In [ ]:
df.info()
df.isnull().sum()

Just plotting a few distributions to see what we are working with.

In [ ]:
df['completion_rate'].hist(bins=20)
plt.title('Completion Rate')
plt.show()

df['avg_feedback_score'].hist(bins=20)
plt.title('Feedback Score')
plt.show()

### 2. Defining Instructor Effectiveness
I need to come up with a score. I'll combine the learner outcomes, engagement, and the feedback.
I'll use MinMaxScaler so all values are on a similar scale before adding them up.

Outcomes: completion_rate, avg_score_improvement
Engagement: avg_watch_time, assignment_submission_rate
Feedback: avg_feedback_score

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
cols = ['avg_score_improvement', 'avg_feedback_score']
df_scaled = df.copy()
df_scaled[cols] = scaler.fit_transform(df[cols])

# calculating a weighted sum for effectiveness
df['score'] = (
    0.2 * df['completion_rate'] + 
    0.2 * df_scaled['avg_score_improvement'] +
    0.15 * df['avg_watch_time'] +
    0.15 * df['assignment_submission_rate'] +
    0.3 * df_scaled['avg_feedback_score']
)

# finding the cutoffs for low, medium, high
cutoffs = df['score'].quantile([0.33, 0.67]).values

def get_tier(s):
    if s <= cutoffs[0]: return 'Low'
    elif s <= cutoffs[1]: return 'Medium'
    else: return 'High'
    
df['tier'] = df['score'].apply(get_tier)
df['tier'].value_counts()

### 3. Aggregating to instructor level
Since instructors teach multiple batches, I will just take the average of all their batches. That seems like the most straightforward way to evaluate them overall.

In [ ]:
instructor_df = df.groupby('instructor_id').mean(numeric_only=True).reset_index()

# recalculate the tier for the instructor based on their average score
agg_cutoffs = instructor_df['score'].quantile([0.33, 0.67]).values

def get_agg_tier(s):
    if s <= agg_cutoffs[0]: return 'Low'
    elif s <= agg_cutoffs[1]: return 'Medium'
    else: return 'High'

instructor_df['instructor_tier'] = instructor_df['score'].apply(get_agg_tier)

# check how many batches they taught
counts = df.groupby('instructor_id').size().reset_index(name='batch_count')
instructor_df = pd.merge(instructor_df, counts, on='instructor_id')

instructor_df.head()

### 4. Building ML Model
I'll try a Random Forest classifier to predict the instructor tier from the aggregated data. It's usually a solid choice for things like this without needing too much tuning.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

label_map = {'Low': 0, 'Medium': 1, 'High': 2}
y = instructor_df['instructor_tier'].map(label_map)

feats = [
    'completion_rate', 'avg_score_improvement', 'avg_quiz_score', 
    'dropout_rate', 'avg_watch_time', 'assignment_submission_rate', 
    'forum_activity_rate', 'avg_feedback_score', 'feedback_response_rate',
    'batch_count'
]
X = instructor_df[feats]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

### 5. Evaluation
Testing how the model performed.

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, cmap='Blues', xticklabels=['Low', 'Medium', 'High'], yticklabels=['Low', 'Medium', 'High'])
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

### 6. Interpretation
Let's check which features the model thought were most important.

In [ ]:
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.bar(range(len(feats)), importances[idx])
plt.xticks(range(len(feats)), [feats[i] for i in idx], rotation=45, ha='right')
plt.show()

### Mandatory Questions

**1. Which features most influenced instructor effectiveness, and why?**
The features that mattered most were feedback score, score improvement, and completion rate. This makes sense because those were the exact things I gave the most weight to when calculating the score in the first place.

**2. Which variables could be misleading or confounded?**
Forum activity and watch time could be misleading. A long watch time might just mean the video was left playing or the students were confused and had to rewatch it, not necessarily that the instructor was good. Also feedback scores can be biased if the course is just naturally easier.

**3. How could this model fail in real-world usage?**
If a new type of course doesn't have homework or forums, the model might penalize that instructor. Also, instructors who have only taught 1 or 2 batches might get predicted poorly compared to someone who has taught 50 batches, since we're just comparing averages.

**4. What additional data would you want to improve this analysis?**
It would be great to know what type of course it is (like programming vs soft skills) so we can compare instructors teaching similar things. Also knowing how experienced the instructor is overall would give good context.

**5. Should this model be used for instructor performance evaluation? Why or why not?**
I wouldn't use it directly for firing or promotions. It's a good tool to maybe flag instructors who need some help, but if people know this model is used for their salary, they might just start giving easier quizzes to get better feedback scores. It should just be a reference point.